# 04a · Modèle 1 : durée de livraison, **tous les modèles de régression**
**Projet : E-Commerce Intelligence Platform** · CRISP-DM, phase 4 (Modeling) et phase 5 (Evaluation)

**Objectif** : prédire, dès l'achat, le nombre de jours entre la commande et la livraison, avec une erreur plus faible que la promesse d'Olist **et** que des baselines simples.

**Démarche**
1. Baselines : promesse d'Olist, médiane globale, médiane par état du client.
2. **Tous les modèles de régression** sont entraînés et comparés sur le bloc valid :
   linéaires (Régression linéaire, Ridge, Lasso, ElasticNet, BayesianRidge, Huber, GLM Gamma), voisins (KNN), noyaux (SVR, LinearSVR), réseau de neurones (MLP), arbres (Decision Tree, Random Forest, Extra Trees, Gradient Boosting, HistGradientBoosting, AdaBoost) et boosting moderne (XGBoost, LightGBM, CatBoost, en variantes L2, L1 et log).
3. Réglage des meilleurs modèles de boosting par validation croisée **temporelle** sur le train.
4. **Mélange** des 3 meilleurs modèles réglés (moyenne des prédictions).
5. Évaluation finale **une seule fois** sur le bloc test, analyse des erreurs, dérive temporelle, importance des variables et **simulateur de promesse**.

> Données : dataset Kaggle créé à partir de la sortie du notebook 03 (ajouter `e-commerce-dataset-model-1` en entrée). Exécuter les cellules dans l'ordre. Durée indicative : 15 à 30 minutes.

## 1. Imports et paramètres

In [ ]:
import os, glob, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import lightgbm as lgb
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor,
                              HistGradientBoostingRegressor, AdaBoostRegressor)
from sklearn.linear_model import (LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge,
                                  HuberRegressor, GammaRegressor)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error
from sklearn.model_selection import ParameterSampler, TimeSeriesSplit
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.svm import SVR, LinearSVR
from sklearn.tree import DecisionTreeRegressor
try:
    import xgboost as xgb; HAS_XGB = True
except Exception:
    HAS_XGB = False
try:
    from catboost import CatBoostRegressor; HAS_CAT = True
except Exception:
    HAS_CAT = False
print("XGBoost :", HAS_XGB, "| CatBoost :", HAS_CAT)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

SEED = 42
rng = np.random.default_rng(SEED)
FAST = os.environ.get("FAST", "0") == "1"          # mode rapide (tests uniquement)
OUT_DIR = os.environ.get("OUT_DIR", "/kaggle/working/models")
os.makedirs(OUT_DIR, exist_ok=True)

USE_PROMISE = None             # None : choix automatique (valid) ; True ou False pour forcer
SKIP_MODELS = set()            # ex. {"SVR (rbf)", "KNN"} pour gagner du temps
N_EST   = 120 if FAST else 500        # arbres des modèles de boosting
RF_TREES = 30 if FAST else 150
GB_TREES = 40 if FAST else 150
SLOW_SUB = 1500 if FAST else 12000    # lignes d'entraînement pour les modèles en O(n²) (SVR)
N_ITER_SEARCH = 2 if FAST else 8

# Chemins des jeux de données Kaggle (créés à partir de la sortie du notebook 03)
DATA_PATHS = {
    "model1_delivery_duration.csv": "/kaggle/input/datasets/oussamafriha/e-commerce-dataset-model-1/model1_delivery_duration.csv",
    "model2_late_risk.csv": "/kaggle/input/datasets/oussamafriha/e-commerce-dataset-model-2/model2_late_risk.csv",
    "model3_customer_segmentation.csv": "/kaggle/input/datasets/oussamafriha/e-commerce-dataset-model-3/model3_customer_segmentation.csv",
}

def find_file(name):
    """DATA_DIR (si défini), puis le chemin Kaggle ci-dessus, puis une recherche dans /kaggle/input."""
    cands = []
    if os.environ.get("DATA_DIR"):
        cands.append(os.path.join(os.environ["DATA_DIR"], name))
    cands.append(DATA_PATHS.get(name, ""))
    cands += glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    for c in cands:
        if c and os.path.exists(c):
            return c
    raise FileNotFoundError(f"{name} introuvable : ajoute le dataset correspondant en entrée du notebook (Add Input).")

## 2. Chargement et blocs temporels

In [ ]:
m1 = pd.read_csv(find_file("model1_delivery_duration.csv"), parse_dates=["order_purchase_timestamp"])

FEATURES_NUMERIC = [
    "nb_items", "multi_seller", "total_price", "total_freight", "total_weight_g", "total_volume_cm3",
    "weight_imputed", "freight_ratio", "limit_days", "distance_km", "distance_imputed", "same_state",
    "customer_lat", "customer_lng", "seller_lat", "seller_lng", "payment_installments",
    "purchase_month", "purchase_dow", "purchase_hour", "is_weekend", "is_black_friday_week", "orders_last_7d",
    "seller_n_past", "seller_mean_days", "seller_late_rate", "route_n_past", "route_mean_days", "route_late_rate",
    "log_total_price", "log_total_freight", "log_total_weight_g", "log_total_volume_cm3", "log_distance_km",
]
CATS = ["customer_state", "seller_state", "categorie_grouped", "payment_type"]
PROMISE = ["promised_days"]
FEATURES_CORE = FEATURES_NUMERIC + CATS

needed = FEATURES_CORE + PROMISE + ["delivery_days", "log_delivery_days", "split",
         "baseline_global_median_days", "baseline_state_median_days", "order_id"]
assert not [c for c in needed if c not in m1.columns], "Colonnes absentes"

tr, va, te = [m1[m1["split"] == s].sort_values("order_purchase_timestamp").reset_index(drop=True)
              for s in ["train", "valid", "test"]]
print(f"train {len(tr):,} | valid {len(va):,} | test {len(te):,}")
print("Durée médiane (jours) :", {n: round(d['delivery_days'].median(), 2) for n, d in [("train", tr), ("valid", va), ("test", te)]})
ytr, yva, yte = tr["delivery_days"].values, va["delivery_days"].values, te["delivery_days"].values

## 3. Baselines
Évaluées sur **valid** : la promesse d'Olist, la médiane globale du train, la médiane par état du client (train). Un modèle n'a de valeur que s'il bat nettement ces trois points de comparaison.

In [ ]:
def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return {"MAE": mean_absolute_error(y, p), "RMSE": float(np.sqrt(mean_squared_error(y, p))),
            "R2": r2_score(y, p), "MedAE": median_absolute_error(y, p), "Biais": float(np.mean(p - y))}

BASELINES = [("Promesse d'Olist", "promised_days"), ("Médiane globale (train)", "baseline_global_median_days"),
             ("Médiane par état (train)", "baseline_state_median_days")]
base_rows = [{"modèle": n, "famille": "Baseline", **metrics(yva, va[c])} for n, c in BASELINES]
display(pd.DataFrame(base_rows).set_index("modèle"))

## 4. Préparation des variables pour chaque famille de modèles
Quatre représentations des mêmes variables (fonction `build_matrices`) : `lin` (plafonnement, standardisation, one-hot) pour les modèles linéaires, voisins, SVM et réseaux ; `ord` (catégories encodées en entiers) pour les arbres, XGBoost et le boosting sklearn ; `lgb` et `cat` pour la gestion native des catégories de LightGBM et CatBoost.

In [ ]:
class Clipper(BaseEstimator, TransformerMixin):
    """Plafonne chaque variable aux quantiles q et 1-q du jeu d'ajustement (limite l'effet des extrêmes)."""
    def __init__(self, q=0.001): self.q = q
    def fit(self, X, y=None):
        X = np.asarray(X, float)
        self.lo_, self.hi_ = np.nanquantile(X, self.q, axis=0), np.nanquantile(X, 1 - self.q, axis=0)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, float), self.lo_, self.hi_)

def take(X, idx):
    return X.iloc[idx] if hasattr(X, "iloc") else X[idx]

def make_converter(feats, ref):
    """Catégories au format pandas « category » (LightGBM), niveaux fixés sur le jeu de référence."""
    cats = [c for c in CATS if c in feats]
    levels = {c: sorted(ref[c].astype(str).unique()) for c in cats}
    def conv(df):
        X = df[feats].copy()
        for c in cats:
            X[c] = pd.Categorical(X[c].astype(str), categories=levels[c])
        return X
    return conv

def build_matrices(feats, fit_df, dfs):
    """Quatre représentations des mêmes variables, ajustées sur fit_df :
    lin = numériques plafonnées + standardisées + one-hot (modèles linéaires, voisins, SVM, réseaux)
    ord = numériques + catégories encodées en entiers (arbres, forêts, boosting sklearn et XGBoost)
    lgb = DataFrame avec dtype category (LightGBM)
    cat = DataFrame avec catégories en texte (CatBoost)"""
    num = [c for c in feats if c not in CATS]
    cats = [c for c in feats if c in CATS]
    lin = ColumnTransformer([("num", Pipeline([("clip", Clipper()), ("sc", StandardScaler())]), num),
                             ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), cats)],
                            sparse_threshold=0).fit(fit_df[feats])
    ordt = ColumnTransformer([("num", "passthrough", num),
                              ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cats)]).fit(fit_df[feats])
    conv = make_converter(feats, fit_df)
    M = {"lin": {}, "ord": {}, "lgb": {}, "cat": {}}
    for n, d in dfs.items():
        M["lin"][n] = np.asarray(lin.transform(d[feats]), dtype=float)
        M["ord"][n] = np.asarray(ordt.transform(d[feats]), dtype=float)
        M["lgb"][n] = conv(d)
        X = d[feats].copy()
        for c in cats:
            X[c] = X[c].astype(str)
        M["cat"][n] = X
    return M

print("Fonctions de préparation prêtes.")

## 5. Registre des modèles de régression
Chaque modèle est décrit par sa famille, la représentation qu'il utilise, un éventuel sous-échantillonnage (SVR) et une grille d'hyperparamètres (boosting). Les modèles dont la bibliothèque est absente sont ignorés.

In [ ]:
LGB_BASE = dict(n_estimators=N_EST, learning_rate=0.05, num_leaves=63, min_child_samples=50, subsample=0.8,
                subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, random_state=SEED, n_jobs=-1, verbose=-1)
XGB_BASE = dict(n_estimators=N_EST, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                min_child_weight=5, tree_method="hist", random_state=SEED, n_jobs=-1)
CAT_BASE = dict(iterations=N_EST, learning_rate=0.08, depth=6, random_seed=SEED, verbose=0, thread_count=-1)

ne = [100, 150] if FAST else [300, 500, 800]
GRID_LGB = {"num_leaves": [15, 31, 63, 127], "learning_rate": [0.02, 0.03, 0.05, 0.08], "min_child_samples": [20, 50, 100, 200],
            "subsample": [0.7, 0.8, 0.9, 1.0], "colsample_bytree": [0.5, 0.7, 0.9], "reg_lambda": [0, 1, 5, 20], "n_estimators": ne}
GRID_XGB = {"max_depth": [3, 4, 6, 8], "learning_rate": [0.02, 0.03, 0.05, 0.08], "min_child_weight": [1, 5, 20, 50],
            "subsample": [0.7, 0.8, 0.9, 1.0], "colsample_bytree": [0.5, 0.7, 0.9], "reg_lambda": [0.5, 1, 5, 20], "n_estimators": ne}
GRID_CAT = {"depth": [4, 6, 8], "learning_rate": [0.03, 0.05, 0.08, 0.12], "l2_leaf_reg": [1, 3, 10, 30], "iterations": ne}

def build_registry():
    R = {}
    def add(name, family, prep, make, sub=None, target="raw", grid=None):
        R[name] = {"family": family, "prep": prep, "make": make, "sub": sub, "target": target, "grid": grid}

    # Linéaires et GLM
    add("Régression linéaire", "Linéaire", "lin", lambda p=None: LinearRegression())
    add("Ridge", "Linéaire", "lin", lambda p=None: Ridge(alpha=10.0))
    add("Lasso", "Linéaire", "lin", lambda p=None: Lasso(alpha=0.01, max_iter=5000))
    add("ElasticNet", "Linéaire", "lin", lambda p=None: ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000))
    add("BayesianRidge", "Linéaire", "lin", lambda p=None: BayesianRidge())
    add("Huber", "Linéaire robuste", "lin", lambda p=None: HuberRegressor(alpha=1e-3, max_iter=300))
    add("GLM Gamma", "GLM", "lin", lambda p=None: GammaRegressor(alpha=1e-3, max_iter=500))
    # Voisins, noyaux, réseaux
    add("KNN", "Voisins", "lin", lambda p=None: KNeighborsRegressor(n_neighbors=25, weights="distance", n_jobs=-1))
    add("SVR (rbf)", "Noyau", "lin", lambda p=None: SVR(C=10.0, epsilon=0.5), sub=SLOW_SUB)
    add("LinearSVR", "Noyau", "lin", lambda p=None: LinearSVR(C=1.0, epsilon=0.0, max_iter=5000, random_state=SEED))
    add("MLP", "Réseau de neurones", "lin", lambda p=None: MLPRegressor(hidden_layer_sizes=(128, 64), early_stopping=True,
        max_iter=60 if FAST else 200, random_state=SEED), target="log")
    # Arbres et ensembles sklearn
    add("Decision Tree", "Arbre", "ord", lambda p=None: DecisionTreeRegressor(max_depth=10, min_samples_leaf=50, random_state=SEED))
    add("Random Forest", "Bagging", "ord", lambda p=None: RandomForestRegressor(n_estimators=RF_TREES, max_depth=16,
        min_samples_leaf=20, max_features=0.5, n_jobs=-1, random_state=SEED))
    add("Extra Trees", "Bagging", "ord", lambda p=None: ExtraTreesRegressor(n_estimators=RF_TREES, max_depth=18,
        min_samples_leaf=20, max_features=0.7, n_jobs=-1, random_state=SEED))
    add("Gradient Boosting (sklearn)", "Boosting", "ord", lambda p=None: GradientBoostingRegressor(n_estimators=GB_TREES,
        learning_rate=0.08, max_depth=4, subsample=0.8, random_state=SEED))
    add("HistGradientBoosting", "Boosting", "ord", lambda p=None: HistGradientBoostingRegressor(max_iter=2 * GB_TREES,
        learning_rate=0.06, random_state=SEED))
    add("AdaBoost", "Boosting", "ord", lambda p=None: AdaBoostRegressor(DecisionTreeRegressor(max_depth=4), n_estimators=60 if FAST else 100,
        learning_rate=0.5, random_state=SEED))
    # Boosting moderne
    def mk_lgb(variant):
        def make(p=None):
            pp = {**LGB_BASE, **(p or {})}
            if variant == "L1": pp["objective"] = "regression_l1"
            return lgb.LGBMRegressor(**pp)
        return make
    add("LightGBM (L2)", "Boosting moderne", "lgb", mk_lgb("L2"), grid=GRID_LGB)
    add("LightGBM (L1)", "Boosting moderne", "lgb", mk_lgb("L1"), grid=GRID_LGB)
    add("LightGBM (log)", "Boosting moderne", "lgb", mk_lgb("L2"), target="log", grid=GRID_LGB)
    if HAS_XGB:
        add("XGBoost (L2)", "Boosting moderne", "ord", lambda p=None: xgb.XGBRegressor(**{**XGB_BASE, **(p or {})}), grid=GRID_XGB)
        add("XGBoost (L1)", "Boosting moderne", "ord", lambda p=None: xgb.XGBRegressor(**{**XGB_BASE, **(p or {}),
            "objective": "reg:absoluteerror"}), grid=GRID_XGB)
    if HAS_CAT:
        add("CatBoost (RMSE)", "Boosting moderne", "cat", lambda p=None: CatBoostRegressor(**{**CAT_BASE, **(p or {})}), grid=GRID_CAT)
        add("CatBoost (MAE)", "Boosting moderne", "cat", lambda p=None: CatBoostRegressor(**{**CAT_BASE, **(p or {}),
            "loss_function": "MAE"}), grid=GRID_CAT)
    return {k: v for k, v in R.items() if k not in SKIP_MODELS}

REG = build_registry()
print(len(REG), "modèles :", ", ".join(REG))

def fit_one(spec, M, y, params=None, rows=None):
    """Entraîne un modèle sur la représentation spec['prep'] ; rows = indices (validation croisée)."""
    X = M[spec["prep"]]["fit"]
    yy = y
    if rows is not None:
        X, yy = take(X, rows), y[rows]
    if spec["sub"] and len(yy) > spec["sub"]:
        idx = np.sort(rng.choice(len(yy), spec["sub"], replace=False))
        X, yy = take(X, idx), yy[idx]
    est = spec["make"](params)
    target = np.log1p(yy) if spec["target"] == "log" else yy
    if spec["prep"] == "cat":
        est.fit(X, target, cat_features=[c for c in CATS if c in X.columns])
    else:
        est.fit(X, target)
    return est

def predict_one(spec, est, X):
    p = np.asarray(est.predict(X), float).ravel()
    p = np.expm1(p) if spec["target"] == "log" else p
    return np.clip(p, 0.0, None)

## 6. Faut-il utiliser la promesse d'Olist comme variable ?
`promised_days` est connue à l'achat, mais elle reflète les règles internes d'Olist. On compare deux modèles représentatifs (Ridge et LightGBM L1), **avec et sans** la promesse, sur valid.

In [ ]:
sets = {"sans promesse": FEATURES_CORE, "avec promesse": FEATURES_CORE + PROMISE}
probe = []
for set_name, feats in sets.items():
    Mp = build_matrices(feats, tr, {"fit": tr, "va": va})
    for name in ["Ridge", "LightGBM (L1)"]:
        spec = REG[name]
        est = fit_one(spec, {k: {"fit": Mp[k]["fit"]} for k in Mp}, ytr)
        probe.append({"modèle": name, "variables": set_name, **metrics(yva, predict_one(spec, est, Mp[spec["prep"]]["va"]))})
probe = pd.DataFrame(probe)
display(probe)

if USE_PROMISE is None:
    use_promise = probe[probe["modèle"] == "LightGBM (L1)"].sort_values("MAE").iloc[0]["variables"] == "avec promesse"
else:
    use_promise = bool(USE_PROMISE)
FEATS = FEATURES_CORE + (PROMISE if use_promise else [])
gain = probe[(probe["modèle"] == "LightGBM (L1)") & (probe["variables"] == "sans promesse")]["MAE"].iloc[0] \
     - probe[(probe["modèle"] == "LightGBM (L1)") & (probe["variables"] == "avec promesse")]["MAE"].iloc[0]
print(f"Variables retenues : {'avec' if use_promise else 'sans'} promesse | apport de la promesse (LightGBM L1) : {gain:+.3f} jour de MAE")

## 7. Entraînement et comparaison de tous les modèles (bloc valid)
Chaque modèle est entraîné sur le **train** et évalué sur **valid**. Un échec d'un modèle n'arrête pas la comparaison. La colonne `temps (s)` donne le coût d'entraînement.

In [ ]:
M = build_matrices(FEATS, tr, {"fit": tr, "va": va, "te": te})
fitted, pred_va, all_rows = {}, {}, []
for name, spec in REG.items():
    t0 = time.time()
    try:
        est = fit_one(spec, M, ytr)
        p = predict_one(spec, est, M[spec["prep"]]["va"])
        fitted[name], pred_va[name] = est, p
        all_rows.append({"modèle": name, "famille": spec["family"], **metrics(yva, p), "temps (s)": time.time() - t0})
        print(f"  {name:30s} MAE valid = {all_rows[-1]['MAE']:.3f}   ({all_rows[-1]['temps (s)']:.0f} s)")
    except Exception as e:
        print(f"  {name:30s} ÉCHEC : {type(e).__name__}: {str(e)[:80]}")

comparison = pd.DataFrame(base_rows + all_rows).sort_values("MAE").reset_index(drop=True)
comparison.insert(0, "rang", comparison.index + 1)
display(comparison)

## 8. Visualisation de la comparaison

In [ ]:
plot_df = comparison.sort_values("MAE", ascending=False)
colors = ["tab:red" if f == "Baseline" else "tab:blue" for f in plot_df["famille"]]
fig, ax = plt.subplots(1, 2, figsize=(18, 0.38 * len(plot_df) + 2))
ax[0].barh(plot_df["modèle"], plot_df["MAE"], color=colors); ax[0].set_xlabel("MAE sur valid (jours)")
ax[0].set_title("Erreur absolue moyenne par modèle (rouge = baselines)")
fam = comparison[comparison["famille"] != "Baseline"].groupby("famille")["MAE"].min().sort_values()
fam.plot.barh(ax=ax[1], color="tab:green"); ax[1].invert_yaxis(); ax[1].set_xlabel("Meilleure MAE de la famille (jours)")
ax[1].set_title("Meilleur modèle de chaque famille")
plt.tight_layout(); plt.show()

fit_time = comparison[comparison["famille"] != "Baseline"].sort_values("temps (s)", ascending=False).head(8)
display(fit_time[["modèle", "temps (s)", "MAE"]])

## 9. Réglage des meilleurs modèles de boosting
Les **3 meilleurs modèles disposant d'une grille** (LightGBM, XGBoost, CatBoost) sont réglés par recherche aléatoire avec `TimeSeriesSplit` sur le train : chaque pli est évalué sur une période **postérieure** à son entraînement.

In [ ]:
tunable = [n for n in comparison["modèle"] if n in REG and REG[n]["grid"] is not None and n in fitted][:3]
print("Modèles à régler :", tunable)
tscv = TimeSeriesSplit(n_splits=3)
best_params, tuned_va, tuned_est, search_log = {}, {}, {}, []
n_tr = len(tr)
for name in tunable:
    spec = REG[name]
    results = []
    for params in ParameterSampler(spec["grid"], n_iter=N_ITER_SEARCH, random_state=SEED):
        maes = []
        for a, b in tscv.split(np.arange(n_tr)):
            est = fit_one(spec, M, ytr, params, rows=a)
            p = predict_one(spec, est, take(M[spec["prep"]]["fit"], b))
            maes.append(mean_absolute_error(ytr[b], p))
        results.append((float(np.mean(maes)), params))
    results.sort(key=lambda r: r[0])
    best_params[name] = results[0][1]
    est = fit_one(spec, M, ytr, results[0][1])
    tuned_est[name] = est
    tuned_va[name + " (réglé)"] = predict_one(spec, est, M[spec["prep"]]["va"])
    search_log.append({"modèle": name, "MAE_cv": results[0][0],
                       "MAE valid défaut": comparison.loc[comparison["modèle"] == name, "MAE"].iloc[0],
                       "MAE valid réglé": mean_absolute_error(yva, tuned_va[name + " (réglé)"]), "paramètres": results[0][1]})
pd.set_option("display.max_colwidth", 200)
display(pd.DataFrame(search_log))

## 10. Mélange des 3 meilleurs modèles
Moyenne des prédictions des 3 modèles réglés. Un mélange bat souvent chacun de ses composants, car les erreurs de modèles différents se compensent en partie.

In [ ]:
blend_va = np.mean([tuned_va[n + " (réglé)"] for n in tunable], axis=0)
final_rows = [{"modèle": k, "famille": "Boosting réglé", **metrics(yva, v)} for k, v in tuned_va.items()]
final_rows.append({"modèle": "Mélange (moyenne des 3)", "famille": "Mélange", **metrics(yva, blend_va)})
final_valid = pd.DataFrame(final_rows).sort_values("MAE").reset_index(drop=True)
display(final_valid)
best_final = final_valid.iloc[0]["modèle"]
print("Meilleure configuration sur valid :", best_final)

## 11. Évaluation finale sur le bloc test
Les modèles finalistes sont évalués **une seule fois** sur le test, avec le modèle entraîné sur le train. Ensuite, la configuration retenue sur valid est **réentraînée sur train + valid** (version destinée à l'usage opérationnel) et évaluée à son tour.

In [ ]:
tuned_te = {n + " (réglé)": predict_one(REG[n], tuned_est[n], M[REG[n]["prep"]]["te"]) for n in tunable}
blend_te = np.mean([tuned_te[n + " (réglé)"] for n in tunable], axis=0)

test_rows = [{"modèle": n, **metrics(yte, te[c])} for n, c in BASELINES]
test_rows += [{"modèle": k, **metrics(yte, v)} for k, v in tuned_te.items()]
test_rows.append({"modèle": "Mélange (moyenne des 3)", **metrics(yte, blend_te)})

# Réentraînement sur train + valid de la configuration retenue
trva = pd.concat([tr, va], ignore_index=True)
ytrva = trva["delivery_days"].values
M2 = build_matrices(FEATS, trva, {"fit": trva, "te": te})
components = tunable if best_final.startswith("Mélange") else [best_final.replace(" (réglé)", "")]
refit = {n: fit_one(REG[n], M2, ytrva, best_params[n]) for n in components}
p_te_final = np.mean([predict_one(REG[n], refit[n], M2[REG[n]["prep"]]["te"]) for n in components], axis=0)
test_rows.append({"modèle": f"FINAL : {best_final} (train + valid)", **metrics(yte, p_te_final)})

test_res = pd.DataFrame(test_rows).set_index("modèle")
display(test_res)
mae_model = test_res.iloc[-1]["MAE"]
for n, _ in BASELINES:
    b = test_res.loc[n, "MAE"]
    print(f"Gain de MAE vs {n:26s}: {b - mae_model:6.2f} jours ({(b - mae_model) / b * 100:5.1f} %)")

## 12. Analyse des erreurs (bloc test, modèle final)

In [ ]:
err = te[["order_id", "customer_state", "distance_km", "order_purchase_timestamp", "promised_days",
          "baseline_state_median_days"]].copy()
err["y"], err["pred"] = yte, p_te_final
err["e_model"] = (err["pred"] - err["y"]).abs()
err["e_state"] = (err["baseline_state_median_days"] - err["y"]).abs()
err["e_promise"] = (err["promised_days"] - err["y"]).abs()
err["mois"] = err["order_purchase_timestamp"].dt.to_period("M").astype(str)

fig, ax = plt.subplots(2, 2, figsize=(16, 10))
err.groupby("mois")[["e_model", "e_state", "e_promise"]].mean().plot(marker="o", ax=ax[0, 0], title="MAE par mois d'achat (test)")
ax[0, 0].set_ylabel("jours"); ax[0, 0].legend(["Modèle final", "Médiane par état", "Promesse d'Olist"])
top = err["customer_state"].value_counts().head(12).index
err[err["customer_state"].isin(top)].groupby("customer_state")[["e_model", "e_state"]].mean().sort_values("e_model").plot.bar(
    ax=ax[0, 1], title="MAE par état du client (12 états les plus fréquents)")
ax[0, 1].legend(["Modèle final", "Médiane par état"])
ax[1, 0].hexbin(err["y"].clip(upper=60), err["pred"].clip(upper=60), gridsize=40, cmap="Blues", mincnt=1, bins="log")
ax[1, 0].plot([0, 60], [0, 60], "r--"); ax[1, 0].set_xlabel("Durée réelle (j, tronquée à 60)"); ax[1, 0].set_ylabel("Durée prédite (j)")
ax[1, 0].set_title("Prédit vs réel")
sns.histplot((err["pred"] - err["y"]).clip(-30, 30), bins=60, ax=ax[1, 1]); ax[1, 1].axvline(0, color="red", ls="--")
ax[1, 1].set_title("Résidus (prédit - réel, jours)")
plt.tight_layout(); plt.show()

err["tranche_reelle"] = pd.cut(err["y"], [0, 7, 14, 21, 30, 60, 1000], labels=["<=7", "8-14", "15-21", "22-30", "31-60", ">60"])
display(err.groupby("tranche_reelle").apply(lambda g: pd.Series({"n": len(g), "MAE": g["e_model"].mean(), "biais": (g["pred"] - g["y"]).mean()})))

## 13. Dérive temporelle

In [ ]:
p_va_best = blend_va if best_final.startswith("Mélange") else tuned_va[best_final]
display(pd.DataFrame({
    "bloc": ["valid (modèle train)", "test (modèle train + valid)"],
    "durée réelle médiane": [np.median(yva), np.median(yte)],
    "durée prédite médiane": [np.median(p_va_best), np.median(p_te_final)],
    "biais moyen (j)": [np.mean(p_va_best - yva), np.mean(p_te_final - yte)]}))
print("Biais moyen par mois (test) :")
print(err.assign(biais=err["pred"] - err["y"]).groupby("mois")["biais"].mean().round(2).to_string())

## 14. Simulateur de promesse : modèles quantiles
Promettre le **quantile q** de la durée prédite signifie qu'environ une part q des commandes arrive à temps. Le graphique compare, sur le test, la **durée promise moyenne** et le **taux de retard**, avec le point de la promesse actuelle d'Olist (LightGBM quantile entraîné sur train + valid).

In [ ]:
Xq_tr, Xq_te = M2["lgb"]["fit"], M2["lgb"]["te"]
qp = best_params.get("LightGBM (L1)", best_params.get("LightGBM (L2)", {}))
quantiles = [0.5, 0.7, 0.8, 0.9, 0.95] if FAST else [0.5, 0.7, 0.8, 0.85, 0.9, 0.95]
qrows = []
for q in quantiles:
    mq = lgb.LGBMRegressor(**{**LGB_BASE, **qp, "objective": "quantile", "alpha": q}).fit(Xq_tr, ytrva)
    pq = mq.predict(Xq_te)
    qrows.append({"quantile": q, "promesse_moyenne_j": pq.mean(), "taux_retard": float((yte > pq).mean())})
qres = pd.DataFrame(qrows)
olist = {"promesse_moyenne_j": float(te["promised_days"].mean()), "taux_retard": float((yte > te["promised_days"].values).mean())}
display(qres)
print(f"Promesse actuelle d'Olist : {olist['promesse_moyenne_j']:.1f} j en moyenne, taux de retard {olist['taux_retard']*100:.1f} %")

plt.figure(figsize=(8, 5))
plt.plot(qres["promesse_moyenne_j"], qres["taux_retard"] * 100, "o-", label="Promesse = quantile prédit")
for _, r in qres.iterrows():
    plt.annotate(f"q={r['quantile']:.2f}", (r["promesse_moyenne_j"], r["taux_retard"] * 100), textcoords="offset points", xytext=(6, 6))
plt.scatter(olist["promesse_moyenne_j"], olist["taux_retard"] * 100, color="red", s=80, zorder=5, label="Promesse actuelle d'Olist")
plt.xlabel("Durée promise moyenne (jours)"); plt.ylabel("Taux de retard (%)"); plt.title("Compromis promesse / retard (test)")
plt.legend(); plt.show()

## 15. Importance des variables (LightGBM, contributions de type SHAP)

In [ ]:
lgb_interp = lgb.LGBMRegressor(**{**LGB_BASE, **qp, "objective": "regression_l1"}).fit(Xq_tr, ytrva)
sample = Xq_te.sample(min(3000, len(Xq_te)), random_state=SEED)
contrib = lgb_interp.booster_.predict(sample, pred_contrib=True)[:, :-1]
imp = pd.Series(np.abs(contrib).mean(axis=0), index=sample.columns).sort_values(ascending=False)
imp.head(20)[::-1].plot.barh(figsize=(8, 7), title="Contribution moyenne absolue (jours)"); plt.show()
display(imp.head(15).to_frame("contribution moyenne absolue"))
try:
    import shap
    codes = sample.copy()
    for c in [c for c in CATS if c in codes.columns]:
        codes[c] = codes[c].cat.codes
    shap.summary_plot(contrib, codes, max_display=15, show=True)
except Exception as e:
    print("Graphique SHAP non affiché :", type(e).__name__, e)

## 16. Sauvegarde

In [ ]:
joblib.dump({n: refit[n] for n in components}, os.path.join(OUT_DIR, "model1_final_components.joblib"))
pred_valid = va[["order_id", "split", "delivery_days", "promised_days", "baseline_state_median_days"]].assign(pred_days=p_va_best)
pred_test = te[["order_id", "split", "delivery_days", "promised_days", "baseline_state_median_days"]].assign(pred_days=p_te_final)
pd.concat([pred_valid, pred_test]).to_csv(os.path.join(OUT_DIR, "model1_predictions.csv"), index=False)
comparison.to_csv(os.path.join(OUT_DIR, "model1_all_models_valid.csv"), index=False)

summary = {"features": FEATS, "use_promise": bool(use_promise), "final": best_final, "components": components,
           "best_params": {k: {a: (b if isinstance(b, (int, float, str)) else float(b)) for a, b in v.items()} for k, v in best_params.items()},
           "test_metrics": {k: {m: float(v) for m, v in row.items()} for k, row in test_res.to_dict("index").items()},
           "promise_simulator": qres.to_dict("records"), "olist_promise": olist}
with open(os.path.join(OUT_DIR, "model1_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print("Fichiers :", sorted(os.listdir(OUT_DIR)))

## Synthèse du modèle 1
À compléter avec tes résultats :
- **Classement des familles de modèles** (cellules 7 et 8) : lesquelles dominent, et de combien par rapport aux baselines.
- **Promesse d'Olist comme variable** : apport mesuré sur valid (cellule 6).
- **Mélange et réglage** : gain du réglage et du mélange par rapport au meilleur modèle par défaut (cellules 9 et 10).
- **Résultat final sur test** : MAE, RMSE, R² et gain en jours contre chaque baseline (cellule 11).
- **Dérive** : biais sur le test (cellule 13) ; **usage métier** : simulateur de promesse (cellule 14).

Étape suivante : `04b-model2-late-risk`.